[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/01_capture_and_look.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Capture and look

**Level 0 of 5. Starting point: you have a model in hand.**

`tl.trace(model, x)` runs one forward pass and records every operation into a `Trace`. This
notebook is about that one call: what it returns, the input forms it accepts, the summary and its
views, and what a capture restores in the model afterwards.

> **You'll want to know:** how to build an `nn.Module` (PyTorch), how to call it on a tensor
> (PyTorch), and nothing else. If you would rather see the whole core first, run
> [00, TorchLens in five minutes](00_quickstart.ipynb).

In [2]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import tiny_cnn, tiny_mlp  # tiny seeded CPU models; nothing downloads

cnn, image = tiny_cnn()  # a stem, six conv blocks (conv, BatchNorm, ReLU), pooling, a linear head
model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

One call captures the CNN. The full-depth summary folds its six identical blocks into one row,
so the whole model reads in seven lines: where the parameters and the FLOPs sit, module by
module.


In [4]:
trace = tl.trace(cnn, image)
trace.summary(depth="all")

name (type),output,params,(%),fwd flops
stem (Conv2d),"(2, 8, 16, 16)",80,2.1%,77.8k
blocks (Sequential),"(2, 8, 16, 16)",3.6k,95.5%,3.7M
0..5 (ConvBlock ×6),"(2, 8, 16, 16)",3.6k,95.5%,3.7M
conv (Conv2d),"(2, 8, 16, 16)",584,15.5%,593.9k
norm (BatchNorm2d),"(2, 8, 16, 16)",16,0.4%,20.5k
pool (AdaptiveAvgPool2d),"(2, 8, 1, 1)",-,-,4.1k
head (Linear),"(2, 10)",90,2.4%,340


## 1. One verb

`tl.trace` returns a `Trace`. It holds a record for every operation the forward ran, grouped
into the modules they ran in; `trace.params` adds up the parameters and `trace.reconstruct_output()`
rebuilds what the model returned. `print(trace)` is the long form, a full log of the pass; the counts are the
short one.

In [5]:
print(len(trace.ops), "ops |", len(trace.modules), "modules |", trace.params.numel, "parameters")
tuple(trace.reconstruct_output().shape)

23 ops | 23 modules | 3770 parameters


(2, 10)

## 2. The summary and its views

A bare `trace.summary()` is the overview: one row per module or operation, at a depth chosen to
stay under 48 rows, with the output shape, the parameters and their share, and the forward FLOPs.
Three dials change the grain: `depth="all"` (the cell above) shows the whole module tree and folds
repeated siblings into one row (`0..5 (ConvBlock x6)`); `view="compute"` adds MACs and an evidence
column saying how each count was obtained; `level="op"` gives one row per operation, labelled the
way `trace[...]` addresses them. `str()` of any summary is plain text.


In [6]:
trace.summary(view="compute", depth="all")

name (type),output,params,fwd flops,macs,evidence
stem (Conv2d),"(2, 8, 16, 16)",80,77.8k,36.9k,formula_exact
blocks (Sequential),"(2, 8, 16, 16)",3.6k,3.7M,1.8M,estimated
0..5 (ConvBlock ×6),"(2, 8, 16, 16)",3.6k,3.7M,1.8M,estimated
conv (Conv2d),"(2, 8, 16, 16)",584,593.9k,294.9k,formula_exact
norm (BatchNorm2d),"(2, 8, 16, 16)",16,20.5k,0,estimated
pool (AdaptiveAvgPool2d),"(2, 8, 1, 1)",-,4.1k,0,estimated
head (Linear),"(2, 10)",90,340,160,formula_exact


In [7]:
tl.trace(model, x).summary(level="op")  # the MLP: one row per operation

name (type),output,params,(%),fwd flops
linear_1,"(4, 16)",-,-,1.1k
relu_1,"(4, 16)",-,-,64
linear_2,"(4, 4)",-,-,528


## 3. Three ways to hand over an input

A real input is the best one: the shapes, the values and the path through `forward` are all
yours. Without one, pass `input_shape=` and TorchLens synthesizes seeded random values and says so
in the summary. With nothing at all it infers a shape and discloses that too, or tells you what
it needs. These three inputs take the same path through this CNN; a model with input-dependent
branches can take different paths for different values or shapes. A `forward` that takes keyword
arguments gets them through `input_kwargs=`.

In [8]:
real = tl.trace(cnn, image)
shaped = tl.trace(cnn, input_shape=(2, 1, 16, 16))  # your shape, seeded random values, disclosed
inferred = tl.trace(cnn)  # nothing: TorchLens infers a shape (a standard image size here)
by_keyword = tl.trace(model, input_kwargs={"x": x})  # forward(self, x): the same tensor by name
for name, t in [("real input", real), ("input_shape", shaped), ("inferred", inferred)]:
    print(f"{name:>12}: {len(t.ops)} ops, input {tuple(t['input_1'].shape)}")
print(f"{'by keyword':>12}: {len(by_keyword.ops)} ops, input {tuple(by_keyword['input_1'].shape)}")

  real input: 23 ops, input (2, 1, 16, 16)
 input_shape: 23 ops, input (2, 1, 16, 16)
    inferred: 23 ops, input (1, 1, 224, 224)
  by keyword: 3 ops, input (4, 8)


For a Hugging Face model a prompt string is a real input: `tl.trace(lm, "The quick brown fox")`
tokenizes with the model's own tokenizer. Nothing in this notebook downloads, so that line stays
text here; [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) explains the prompt route.

## 4. The one-call entry points

When the summary is all you want, skip the trace. `tl.summary(model, x)` runs a metadata-only
capture in eval mode under no-grad, restores every training flag afterwards and returns the
report. For a picture, draw the trace: `tl.trace(model, x).draw()`, in [03, See it](03_see_it.ipynb).
`tl.summary` takes the same three input forms as `tl.trace`.

In [9]:
tl.summary(model, x)  # the MLP; the header says what the one-call door restored

name (type),output,params,(%),fwd flops
in_proj (Linear),"(4, 16)",144,67.9%,1.1k
relu_1,"(4, 16)",-,-,64
out_proj (Linear),"(4, 4)",68,32.1%,528


## 5. A trace is a pure observer

By default a capture leaves the model as it found it. Put the CNN in train mode, where a forward pass updates every
BatchNorm's running statistics, and trace it: the statistics do not move. Then call the model
plainly once, and they do. The capture snapshots buffers, training flags and random states before
the forward and restores them after. Two exceptions: a lazy module that initializes during the
forward stays initialized, and a change inside a custom attribute object is not undone;
`CaptureOptions(carry_state=True)` opts out of the restoration. The trace's `outcome` says the forward
ran to completion, and `is_pristine` says nothing intervened in it.

In [10]:
stats = cnn.blocks[0].norm.running_mean  # a BatchNorm buffer that train-mode forwards update
cnn.train()
print("before             :", stats[:3].clone())
observed = tl.trace(cnn, image)
print("after tl.trace     :", stats[:3].clone())
cnn(image)  # a plain forward, for comparison
print("after a plain call :", stats[:3].clone())
cnn.eval()
print("outcome:", observed.outcome.status.name, "| pristine:", observed.is_pristine)

before             : tensor([0., 0., 0.])
after tl.trace     : tensor([0., 0., 0.])
after a plain call : tensor([-0.0100,  0.0019, -0.0153])
outcome: COMPLETE | pristine: True


## 6. Where this leaves you

You can capture any model with one call, hand it an input three ways, read the summary at four
grains, and know what a capture restores afterwards. Everything from here is reading the
`Trace` more closely.

**One thing about networks:** parameters and compute live in different places. In the compute
view the stem's 80 parameters do 77.82K FLOPs, about a thousand each, because a convolution
reuses every weight at every pixel; the head's 90 parameters do 340, about four each, because a
linear layer uses each weight once per item. The six blocks hold 3.6K of the 3,770 parameters
and nearly all of the compute. Compute follows resolution, parameters follow width.

**Go deeper:** next level, [02, Address and read](02_address_and_read.ipynb): every operation has
an address and a record behind it. Recipe card:
[`visualize.ipynb`](../../examples/quick/visualize.ipynb). Reference:
[the quick-start guide](../../docs/quickstart.md) for the three ways to give an input and
[the summary reference](../../docs/reference/summary.md) for every column and dial.